# Popups, tooltips & labels

Every layer binds a popup and a tooltip by default: hover for the tooltip, click
for the popup, every property listed as `name: value`. This notebook narrows,
relabels, templates, and styles them — adds permanent labels — and shows why
hostile data stays harmless.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(4)
n = 40
df = pd.DataFrame({
    "lat": 36.05 + rng.normal(0, 0.04, n),
    "lon": -5.45 + rng.normal(0, 0.07, n),
    "site": [f"Station {i:02d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
    "installed": rng.choice(["2019", "2021", "2024"], n),
    "status": rng.choice(["Active", "Idle"], n),
})

## Narrow and relabel

`popup_fields` picks the properties; `popup_names` relabels them, matched by
position. The same pair exists for tooltips. If names cannot be lined up with
fields, swiftmap warns and falls back to the raw column names rather than failing
the render.

In [ ]:
m = Map()
m.add_circle_markers(
    df, name="Stations",
    popup_fields=["site", "reading", "installed"],
    popup_names=["Site", "Reading (ppm)", "Installed"],
    tooltip_fields=["site", "status"],
)
m

## Templates

`{column}` inserts one value; `{*}` expands the default field list — markup
without enumerating every column. Your markup renders as HTML.

In [ ]:
m = Map()
m.add_circle_markers(
    df, name="Stations",
    popup_template="<b>{site}</b><hr>{*}<br><i>field survey 2026</i>",
    popup_fields=["reading", "status"],
    popup_names=["Reading", "Status"],
    popup_max_width=400,
)
m

## Styling the container

In [ ]:
m = Map()
m.add_circle_markers(
    df, name="Stations",
    tooltip_style="background: #1b1e23; color: #eee; border: 1px solid #444; "
                  "font-size: 11px; padding: 4px 8px;",
)
m

## Constant extra fields

Passing a dict as `popup=` / `tooltip=` adds fixed fields to every feature —
provenance notes, units, anything not worth a column:

In [ ]:
m = Map()
m.add_circle_markers(df, name="Stations",
                     popup={"source": "harbour survey", "datum": "WGS84"})
m

## Hostile data stays harmless

Values pulled from the data are **escaped**; only your template markup renders as
HTML. Click the point below: the script tag displays as text instead of executing.
Values landing in an `href` or `src` are additionally checked for a safe URL
scheme. That matters the moment your map shows data you didn't author — uploads,
shared tables, third-party feeds.

In [ ]:
sus = pd.DataFrame({
    "lat": [36.05], "lon": [-5.45],
    "note": ["<script>alert('nope')</script>"],
})
m = Map()
m.add_circle_markers(sus, name="Escaped", radius=12)
m

## Permanent labels

`label=` puts quiet text chips on the map — a column name labels each feature
from its own value, anything else is the literal text. Points anchor at the
point, lines at their middle vertex, areas at their centre. Chips are DOM
elements — built for sites and zones, not point clouds (the point builders warn
past a thousand) — label text is escaped like popup values, and on a time layer
the chips appear and vanish with their features.

In [ ]:
m = Map()
m.add_circle_markers(df.head(12), name="Stations", label="site", radius=8)
m

## Turning them off

`popup=False` / `tooltip=False` per call.

In [ ]:
m = Map()
m.add_circle_markers(df, name="Quiet", popup=False, tooltip=False)
m